# localrag demo: accurate RAG with a sub-1B local model

Pipeline: section-aware PDF ingestion → hybrid retrieval (BM25/SPLADE + dense, RRF fusion, cross-encoder rerank) → agentic control (routing, query decomposition, corrective retrieval) → grammar-constrained answers whose citations are verified against the cited passages. Optional LoRA adapters trained with DPO / GRPO on a verifiable reward.

Everything below is driven by `configs/default.yaml`; the parameter cell only lists overrides.

## Parameters
Every technique is a switch. `None` keeps the default from `configs/default.yaml`.

In [ ]:
# --- model
GENERATOR = "qwen3.5-0.8b"        # "qwen3.5-0.8b" | "qwen3-0.6b"
ANSWER_ADAPTER = None             # None | "dpo" | "grpo"          (needs: python -m localrag train dpo|grpo)
DECOMPOSE_ADAPTER = None          # None | "grpo-decompose"         (needs: python -m localrag train grpo-decompose)

# --- ingestion (changing these requires re-running the ingest cell)
PARSER = "pymupdf"                # "pymupdf" (hand-written) | "docling"
CHILD_SIZE = 0                    # >0: parent-child retrieval with children of this many characters
CONTEXTUALIZE = False             # LLM-written context prepended to every chunk before indexing

# --- retrieval
RETRIEVAL_MODE = "hybrid"         # "dense" | "sparse" | "hybrid"
SPARSE = "bm25"                   # "bm25" | "splade"
RERANKER = "cross_encoder"        # "cross_encoder" (bge-reranker-v2-m3) | "llama" (Qwen3-Reranker) | None

# --- agent
DECOMPOSE = True
CRAG = False                      # corrective RAG: rewrite + re-retrieve when evidence is weak
ANSWER_MODE = "grounded"          # "grounded" | "grounded_evidence" | "freetext" (original prompt-only citations)
VERIFIER = "nli"                  # "none" | "reranker" | "nli"
SELF_CORRECT = False              # regenerate once if most claims are unsupported
MIN_RELEVANCE = 0.0               # refuse below this top rerank score (python -m localrag calibrate)

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

from localrag.config import load_config
from localrag.pipeline import RAGPipeline, ingest

MODELS = {"qwen3.5-0.8b": ("unsloth/Qwen3.5-0.8B-GGUF", "Qwen3.5-0.8B-Q8_0.gguf"),
          "qwen3-0.6b": ("Qwen/Qwen3-0.6B-GGUF", "Qwen3-0.6B-Q8_0.gguf")}
adapters = {step: f"adapters/{name}.gguf" for step, name in
            (("answer", ANSWER_ADAPTER), ("decompose", DECOMPOSE_ADAPTER)) if name}
for path in adapters.values():
    assert Path(path).exists(), f"{path} not found: train it first (see the parameter cell)"

overrides = {
    "llm.repo_id": MODELS[GENERATOR][0], "llm.filename": MODELS[GENERATOR][1], "llm.adapters": adapters,
    "ingest.parser": PARSER, "ingest.child_size": CHILD_SIZE, "ingest.contextualize": CONTEXTUALIZE,
    "retrieval.mode": RETRIEVAL_MODE, "retrieval.sparse": SPARSE,
    "agent.decompose": DECOMPOSE, "agent.crag": CRAG, "agent.answer_mode": ANSWER_MODE,
    "agent.verifier": VERIFIER, "agent.self_correct": SELF_CORRECT, "agent.min_relevance": MIN_RELEVANCE,
    "agent.support_threshold": 0.5 if VERIFIER == "nli" else 0.05,
}
if RERANKER is None:
    overrides["retrieval.reranker_model"] = None
else:
    overrides["retrieval.reranker_backend"] = RERANKER
# non-default ingestion settings get their own index directory
if (PARSER, CHILD_SIZE, CONTEXTUALIZE) != ("pymupdf", 0, False):
    overrides["index_dir"] = f"index/{PARSER}_c{CHILD_SIZE}_ctx{int(CONTEXTUALIZE)}"

cfg = load_config("configs/default.yaml", overrides)
cfg.index_path

## Build the index
Only needed once per ingestion setting, or after the PDFs in `documents/` change.

In [ ]:
if not (cfg.index_path / "chunks.jsonl").exists():
    ingest(cfg)

## Ask questions
`RAGPipeline` starts a `llama-server` for the GGUF model (plus adapters, if any).

In [ ]:
rag = RAGPipeline(cfg)

def ask(q, trace=True):
    ans = rag.answer(q)
    if trace:
        print("action:", ans.action, "| subqueries:", ans.subqueries, "| timings:", ans.timings,
              "| trace:", ans.trace)
        for i, h in enumerate(ans.passages, 1):
            print(f"[{i}] {h.score:.3f} {h.chunk.source} p.{h.chunk.page} | {h.chunk.section[:70]}")
        for c in ans.claims:
            print(f"   supported={c.supported} ({(c.support_score or 0):.2f}) {c.text[:90]} {c.citations}")
    display(Markdown(ans.text))
    return ans

In [ ]:
ask("Hello!", trace=False);

In [ ]:
ask("What must design verification confirm under the quality system regulation?");

A comparison across two documents (multi-hop):

In [ ]:
ask("How do the off-the-shelf software guidance and the cybersecurity guidance differ in what manufacturers must document about third-party software?");

A question the documents do not answer: the system should refuse rather than improvise.

In [ ]:
ask("What are the EU MDR Annex VIII classification rules for software?");

## Compare base vs. RL-tuned answers (optional)
Runs the same questions with and without the trained answer adapters. Skipped when no adapters exist.

In [ ]:
from localrag.llm import release_llms

available = [n for n in ("dpo", "grpo") if Path(f"adapters/{n}.gguf").exists()]
questions = ["What records must be kept for design reviews?",
             "How should a manufacturer validate off-the-shelf software used in a device?"]
if not available:
    print("No adapters found; train with: python -m localrag train dpo  (or grpo)")
else:
    release_llms()
    for name in [None, *available]:
        c = load_config("configs/default.yaml", {**overrides, "llm.adapters": {"answer": f"adapters/{name}.gguf"} if name else {}})
        pipe = RAGPipeline(c)
        for q in questions:
            a = pipe.answer(q)
            display(Markdown(f"**{name or 'base'}**: {q}\n\n{a.text}"))
        release_llms()

## Evaluation results
`python -m localrag eval` runs every variant in `configs/ablations.yaml` on the held-out test split; the table lands in `eval/results/summary.md`.

In [ ]:
summary = Path("eval/results/summary.md")
display(Markdown(summary.read_text() if summary.exists() else "Run `python -m localrag eval` first."))

In [ ]:
release_llms()